# Update Streaming Mode

In [2]:
from langgraph.graph import StateGraph ,START,END
from langchain_ollama import ChatOllama
from langgraph.graph import MessagesState
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import convert_to_messages
from pprint import pprint

class MessagesState(MessagesState):
    topic: str
# Create a simple graph
graph_builder = StateGraph(MessagesState)

# Define nodes
def tell_me_a_joke(state: MessagesState) -> MessagesState:
    """Node that processes in streaming mode"""
    prompt = PromptTemplate(input_variables=["topic"], template="Tell me a joke about {topic}")
    llm = ChatOllama(model="gpt-oss:20b")
    chain = prompt | llm
    return {"messages" : [chain.invoke(input=state["topic"])]}

# Add nodes to graph
graph_builder.add_node("streaming", tell_me_a_joke)

# Add edges
graph_builder.add_edge(START, "streaming")
graph_builder.add_edge("streaming", END)

# Compile the graph
graph = graph_builder.compile()
joke_topic = input("Enter a topic for the joke: ")
for part in graph.stream(
    {"topic": joke_topic},
    stream_mode=["values", "updates", "messages", "custom"],
    version="v2",
):
    if part["type"] == "values":
        print(f"\n{'='*150}\n[VALUES] Full state snapshot:\n{part['data']}\n{'='*150}")
    elif part["type"] == "updates":
        print(f"\n{'='*150}\n[UPDATES] Changed keys from nodes:")
        for node_name, state in part["data"].items():
            print(f"  └─ Node '{node_name}': {state}")
    elif part["type"] == "messages":
        msg, metadata = part["data"]
        if msg.content:
            print(f"[MESSAGES] LLM streaming chunk: {msg.content}\n", end="", flush=True)
    elif part["type"] == "custom":
        print(f"\n{'='*150}\n[CUSTOM] {part['data']}\n{'='*150}")


[VALUES] Full state snapshot:
{'messages': [], 'topic': 'Humans'}
[MESSAGES] LLM streaming chunk: Why
[MESSAGES] LLM streaming chunk:  do
[MESSAGES] LLM streaming chunk:  humans
[MESSAGES] LLM streaming chunk:  always
[MESSAGES] LLM streaming chunk:  talk
[MESSAGES] LLM streaming chunk:  to
[MESSAGES] LLM streaming chunk:  their
[MESSAGES] LLM streaming chunk:  phones
[MESSAGES] LLM streaming chunk: ?


[MESSAGES] LLM streaming chunk: Because
[MESSAGES] LLM streaming chunk:  it
[MESSAGES] LLM streaming chunk: ’s
[MESSAGES] LLM streaming chunk:  the
[MESSAGES] LLM streaming chunk:  only
[MESSAGES] LLM streaming chunk:  thing
[MESSAGES] LLM streaming chunk:  that
[MESSAGES] LLM streaming chunk:  listens
[MESSAGES] LLM streaming chunk:  without
[MESSAGES] LLM streaming chunk:  interrupt
[MESSAGES] LLM streaming chunk: ing
[MESSAGES] LLM streaming chunk: …
[MESSAGES] LLM streaming chunk:  and
[MESSAGES] LLM streaming chunk:  then
[MESSAGES] LLM streaming chunk:  reminds
[MESSAGES] LLM str